# Mutation Testing Demo on Requests Repository

Repository from: https://github.com/psf/requests

## Project Exploration

In [1]:
from pymut4se.api import discover

workspace = discover("../requests-main") # Root of requests

print(workspace.project.name)
print(workspace.statistics())
print(workspace.statistics().as_dict())

requests-main
ProjectStatistics(packages=1, modules=21, chunks=259, tests=347, tested_chunks=20, requirements=19)
{'packages': 1, 'modules': 21, 'original_chunks': 259, 'test_suites': 17, 'test_cases': 347, 'test_links': 399, 'chunks_with_tests': 20, 'requirements': 19}


In [2]:
modules = workspace.find_modules()
chunks = workspace.find_chunks()
print(modules)


[Module(name='docs._themes.flask_theme_support', path='docs/_themes/flask_theme_support.py', id='8591577b…'), Module(name='docs.conf', path='docs/conf.py', id='70978f97…'), Module(name='setup', path='setup.py', id='5c4fbedb…'), Module(name='src.requests.__version__', path='src/requests/__version__.py', id='0a00a2a6…'), Module(name='src.requests._internal_utils', path='src/requests/_internal_utils.py', id='d1d7b970…'), Module(name='src.requests._types', path='src/requests/_types.py', id='dcb99efa…'), Module(name='src.requests.adapters', path='src/requests/adapters.py', id='a34079af…'), Module(name='src.requests.api', path='src/requests/api.py', id='b739ed46…'), Module(name='src.requests.auth', path='src/requests/auth.py', id='a5567752…'), Module(name='src.requests.certs', path='src/requests/certs.py', id='08a521ae…'), Module(name='src.requests.compat', path='src/requests/compat.py', id='9fca0ad4…'), Module(name='src.requests.cookies', path='src/requests/cookies.py', id='6df089f5…'), Mod

### Available mutation operators

In [3]:
for operator in workspace.operators():
    print(operator.name, operator.class_name, operator.description)

add-if-not-null IfNotNullMutation Guard a function body so it only runs for non-null inputs.
arithmetic ArithmeticMutation Replace one arithmetic binary operator.
boolean-replacement BooleanReplacementMutation Invert one boolean literal.
constant-replacement ConstantReplacementMutation Transform one numeric constant.
control-replacement ControlReplacementMutation Swap break and continue.
delete-assignment DeleteAssignmentMutation Delete one assignment statement.
delete-decorator DeleteDecoratorMutation Delete one function decorator.
delete-if-statement DeleteIfStatementMutation Delete one complete if statement.
delete-return DeleteReturnMutation Delete one return statement.
delete-while DeleteWhileMutation Delete one complete while loop.
logical-connector LogicalConnectorMutation Swap and and or in one boolean expression.
optional-parameter-callee OptionalParamCalleeMutation Change an optional parameter default in the called function.
optional-parameter-caller OptionalParamCallerMutati

## Mutant Generation

In [4]:
""" 
In this demo we will generate mutants for all the code chunks that are found in the fastapi source code.
By default workspace.mutate() will apply all the operators and generate mutants of degree 1.
You can set the operators to any of the available operators.
Note: increasing the max_degree will quickly result in a very large number of mutants if you target an entire project. 
To speed un the generation, you can alo chose to only generate mutants for code chunks with test cases by using:
new_mutants = workspace.mutate_chunks_with_tests(workspace.chunks, "all")
"""

targets = [
    *workspace.find_modules("src/requests"), # you can change the module if you prefer testing only one specific module.
    *workspace.find_chunks("_apply_mutation"),
]

new_mutants = workspace.mutate(
    targets,
    operators="all",
    max_degree=1,
)


Generating mutants: 8267 new | chunks processed: 259/259


In [5]:
"""
You can check the number of mutants per mutation operator.
"""
print(workspace.mutant_statistics())

MutantStatistics(total=8267, source_chunks=259, by_degree={1: 8267}, by_type={'arithmetic': 1939, 'boolean_replacement': 99, 'cast_type': 1053, 'constant_replacement': 464, 'control_replacement': 11, 'delete_assignment': 644, 'delete_decorator': 41, 'delete_if_statement': 328, 'delete_return': 246, 'delete_while': 5, 'if_not_null': 351, 'logical_connector': 99, 'optional_param_callee': 101, 'optional_param_caller': 18, 'relational': 2025, 'return_pass': 222, 'swap_arguments': 381, 'unary': 240}, input_executions=0, test_executions=0, failed_tests=0)


## Mutant execution with test suite

In [6]:
"""
Before starting the execution it is worth checking for which mutants there exist test cases.
If no test cases are found for a code chunk, by default the whole test-suite will be executed.
For better performance you can chose to only execute the mutants with test cases.
"""

mutants_with_tests = [
    mutant for mutant in new_mutants
    if mutant.related_test_cases
]
len(mutants_with_tests)

631

In [7]:
test_outputs = workspace.run_tests(
    mutants_with_tests,
    parallel=True,
    max_workers=6,
    timeout_seconds=2,
)

Preparing execution environment...
Execution environment ready: /Users/laura/Desktop/exp_tool/PyMut4SE/requests-main/.pymut4se/venvs/35727b2b4d59c0fb58e5e2d564e15f0a43576d2b5d2800caa7cab193a7424d3b
Executing tests: 631/631


In [8]:
"""
Print the mutation score and the amount of killed and survived mutants.
"""

score = workspace.mutation_score()
print(score)

MutationScore(score=26.27%, killed=109, survived=306, untested=7636, incomplete=0, errors=216)


## Storing mutants & execution in database

In [9]:
from sqlalchemy import create_engine
from sqlalchemy.orm import Session

from pymut4se.model import Base

engine = create_engine("sqlite:///requests.db")
Base.metadata.create_all(engine)

with Session(engine) as session:
    workspace.save(session, commit=True)